# Stat 220, Unit 6 Homework: Inference Without a Formula

`delivery_routes.csv` is the 500 routes from the Unit 4 homework. `minutes` is how
long a route took. Five columns describe the route: `stops`, `packages`, `miles`, `downtown` and
`rain`. Three more get recorded and have nothing to do with how long anything takes:
`van_age_years`, `dispatcher_rating` and `month`. You will need that last fact.

Run the next two cells first.

In [1]:
%pip install -q numpy pandas statsmodels scikit-learn matplotlib

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

routes = pd.read_csv("https://drbob-richardson.github.io/stat220/F2026/data/delivery_routes.csv")
PREDICTORS = ["stops", "packages", "miles", "downtown", "rain",
              "van_age_years", "dispatcher_rating", "month"]
routes.head()

,minutes,stops,packages,miles,downtown,rain,van_age_years,dispatcher_rating,month
0,254.3,49,105,55.6,1,0,8,3,7
1,153.5,24,71,32.0,0,0,5,5,2
2,450.9,73,226,55.4,1,1,7,4,4
3,387.1,71,175,73.2,1,0,1,1,12
4,111.2,12,18,32.1,0,1,7,1,11


**Problem 1.** Fit a forest to the routes, and a regression beside it, and find out which
predicts better.

Part a. Report the held-out $R^2$ for both.

In [3]:
X = routes[PREDICTORS]
y = routes["minutes"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

forest = RandomForestRegressor(n_estimators=300, random_state=0,
                               n_jobs=-1).fit(X_train, y_train)
line = smf.ols("minutes ~ " + " + ".join(PREDICTORS),
               data=routes.loc[X_train.index]).fit()
pred = line.predict(routes.loc[X_test.index])
line_r2 = 1 - ((y_test - pred) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()

print("forest     held-out R2:", round(forest.score(X_test, y_test), 3))
print("regression held-out R2:", round(line_r2, 3))

forest     held-out R2: 0.955
regression held-out R2: 0.963


_Your answer:_ Fitting a forest to our routes data provides us with a held-out R^2 of 0.955. Fitting a regression on the same data gives us a held-out R^2 of 0.963. Based on this we might consider a regression to be the better fitting model, but there is more we need to know first before making that conclusion.



Part b. One of the two can hand you a coefficient with an interval and the other cannot. Say which, and say what you would have to do to get something comparable out of the one that cannot.

_Your answer:_ Regression is also able to provide us with a coefficient with an interval where the forest cannot. If you wanted to get a comparable metric from a random forest, you would want to get an importance measurement to determine which variables are used the most and used consistently (this acts as a coefficient of sorts). We would also need to draw a resample and turn turn it into an interval.



**Problem 2.** Now ask which columns it is using. Run both importance methods and compare them.

Part a. Report both rankings in one table.

In [4]:
impurity = pd.Series(forest.feature_importances_, index=PREDICTORS)
perm = permutation_importance(forest, X_test, y_test, n_repeats=10, random_state=0, n_jobs=-1)
permutation = pd.Series(perm.importances_mean, index=PREDICTORS)

print(pd.DataFrame({"impurity": impurity.round(4),
                    "permutation": permutation.round(4)}).sort_values("impurity",
                                                                      ascending=False))

                   impurity  permutation
stops                0.9127       1.1920
packages             0.0416       0.0327
miles                0.0312       0.0463
month                0.0049       0.0011
van_age_years        0.0043       0.0003
dispatcher_rating    0.0032       0.0003
downtown             0.0010       0.0000
rain                 0.0010       0.0006


_Your answer:_

```
                   impurity  permutation
stops                0.9127       1.1920
packages             0.0416       0.0327
miles                0.0312       0.0463
month                0.0049       0.0011
van_age_years        0.0043       0.0003
dispatcher_rating    0.0032       0.0003
downtown             0.0010       0.0000
rain                 0.0010       0.0006

```





Part b. Both methods put `month` above `downtown` and `rain`. Two of those three columns are real and one is not. Say which, and explain in two or three sentences how a real predictor can finish below a column that means nothing.

_Your answer:_ Of the three columns, the month column is a false predictor as it does not affect how long a route takes the way 'downtown' and 'rain' do. However, month is placed above both downtown and rain for both importance models. This is because month is likely correlated with the effects of packages and stops (ie. more packages and stops are added in December, and for certain holidays). The real predictors are stronger than 'month', but in this case they are not stronger than the other predictors influencing month's effect.



**Problem 3.** A ranking is not the same as an effect. Fit a regression on the same eight columns and see what it says instead.

Part a. Report the coefficient and p-value for `downtown`, `rain` and `month`.

In [5]:
fit = smf.ols("minutes ~ " + " + ".join(PREDICTORS), data=routes).fit()
for c in ["downtown", "rain", "month"]:
    print(f"{c:<10} coef {fit.params[c]:+7.3f}   p {fit.pvalues[c]:.2g}")

downtown   coef  +4.144   p 0.053
rain       coef  +6.699   p 0.0079
month      coef  +0.014   p 0.96


_Your answer:_ All else held equal,'downtown' increases the route time by about 4.14 minutes (positive coefficient effect = 4.144; p-value = 0.053). All else held equal, 'rain' increases route time by about 6.7 minutes (positive coefficeint = 6.699; p-value = 0.0079). All else held equal, 'month' has a much smaller effect on route time of around 0.01 minutes (positive coeffiecient = 0.014) and does not have apparent statistical significance (p-value = 0.96).



Part b. The forest ranked `month` above `rain`. The regression says something different. Explain in three or four sentences what each of the two is measuring, and which one answers the question 'does rain add time to a route'.

_Your answer:_ The regression shows clearly that 'rain' has a greater significance in its effect on route time than 'month', but both importance measurements say otherwise. This is because importance measures how often a variable appears in a tree, thus helping us to identify potentially important factors, while regression measures the influence of one variable on another directly, all else held constant. In the large amount of minutes a route takes, the seven minutes that rain would add is marginal and therefore is likley under-represented in the splits. So using regression, we can determine that rain does significantly add time to the route (p-value < 0.01).



**Problem 4.** A ranking does not tell you whether a column is doing anything at all, or
whether it just happened to land above one that is doing nothing. Resampling does. Resample the
training rows only, and score against the test set you already have.

Part a. Report the point estimate and a 95% bootstrap interval for each column, and say which ones you cannot separate from zero.

In [6]:
rng = np.random.default_rng(6)

def importance_of(X_fit, y_fit, seed):
    f = RandomForestRegressor(n_estimators=60, random_state=0, n_jobs=-1).fit(X_fit, y_fit)
    r = permutation_importance(f, X_test, y_test, n_repeats=4, random_state=seed, n_jobs=-1)
    return pd.Series(r.importances_mean, index=PREDICTORS)

base = importance_of(X_train, y_train, 0)
reps = pd.DataFrame([importance_of(X_train.iloc[take], y_train.iloc[take], k)
                     for k, take in enumerate(rng.integers(0, len(X_train),
                                                           (30, len(X_train))))])

for c in PREDICTORS:
    lo, hi = np.percentile(reps[c], [2.5, 97.5])
    flag = "  cannot separate from zero" if lo <= 0 <= hi else ""
    print(f"{c:<18} {base[c]:+.4f}  [{lo:+.4f}, {hi:+.4f}]{flag}")

stops              +1.2287  [+1.0238, +1.3359]
packages           +0.0275  [+0.0144, +0.0577]
miles              +0.0459  [+0.0310, +0.0599]
downtown           +0.0000  [-0.0009, +0.0010]  cannot separate from zero
rain               +0.0009  [-0.0006, +0.0006]  cannot separate from zero
van_age_years      +0.0008  [-0.0004, +0.0024]  cannot separate from zero
dispatcher_rating  +0.0004  [-0.0024, +0.0012]  cannot separate from zero
month              +0.0007  [-0.0003, +0.0028]  cannot separate from zero


_Your answer:_ After resampling our training rows and scoring them against the test set, we see that the model leans on 'stops' the most (point estimate = 1.2287; 95% BI: 1.0238, 1.3359). The next column leaned on the most is 'miles' (point estimate = 0.0459; 95% BI: 0.0310, 0.599), followed by 'packages' (point estimate = 0.0275; 95% BI: 0.0144, 0.0577). The other columns, 'downtown' (pe = 0; 95% BI; -0.00009, 0.001), 'rain' (pe = 0.0009; 95% BI: -0.0006, 0.0006), 'van_age_years' (pe = 0.0008; 95% BI: -0.0004, 0.0024), 'dispatcher_rating' (pe = 0.0004; 95% BI: -0.0024, 0.0012), and 'month' (pe = 0.0007; 95% BI: -0.0003, 0.0028) have much lower point estimates and cannot be separated from zero based on our data.



Part b. In problem 3 the regression found `rain` real, with about seven extra minutes and a small p-value. Here the importance of `rain` cannot be separated from zero. Both results are correct. Explain in three or four sentences how that happens, and say which of the two you would report to someone asking whether rain slows deliveries.

_Your answer:_ The effect that 'rain' has on route time, in isolation, is best determined using a regression which is capable of isolating predictor effects. Importance measures how often a predictor appears in a tree or forest and tests how much removing a varaible would throw off a prediction. So while this can help determine which variables the model leans on most, it favors the big needle-movers. In the case of rain, the importance interval includes zero because the forest can predict well enough without rain--'stops' taking up most of the responsibility for route time, but that does not mean that rain does not have an effect. If someone asks whether rain slows deliveries, the most accurate answer would come from a regression.



**Problem 5.** No computer for this one. Three colleagues, three claims.

Part a. A colleague fits a forest to customer data and reports that `account_number` is the second most important variable, using the default importance. Write the two or three sentences you would say in response.

_Your answer:_ In response to a colleague claiming that 'account_number' is the second most important variable based on default importance, I would reply, "We should try running a permutation importance on our model. Remember that default impurity importance tests favor variables with lots of unique values, such as the account_number column. Since permutation importance takes correlations into account more it should control for imaginary importance."



Part b. Another colleague bootstraps a sample of 40 patients, gets a 95% interval, and says it shows the treatment works in the wider population. The 40 patients were all recruited from one clinic. Explain what the interval does and does not cover.

_Your answer:_ This specific example seems to simply lack logical reasoning and power. bootstrapping the sample is still the right move with a sample size this small, but it is unreasonable to assume that an unrandom sample of only 40 patients from a single clinic can accurately represent the wider population. It can give great predictions for those 40 patients and may be able to give us a good representation of that specific clinic, but we would need more randomization of subjects from a wider population. As for the 95% bootstrap interval, it represents that across all of the bootstrap sampling, 95% of values fall into for a specific predictor. It does not tell us how effective the treatment is outside of our range of measured values.



Part c. A third colleague resamples the whole dataset, then splits each resample into training and test rows, and reports bootstrap intervals for permutation importance. Every column comes out important. Say what went wrong.

_Your answer:_ In this case, resampling the complete dataset without repsecting a previous 70/30 training and test row separation would undermine the value that comes from the separation. The model is essentially being tested on all of the values over and over and will overfit on the data because each resample split is now arbitrary.

